# Video 4: Crews and Process Types

## What We Will Build
In this notebook, we will build and compare **Sequential and Hierarchical Crews**. You will learn:
- What a Crew is and how to configure it
- Sequential process: tasks execute one after another
- Hierarchical process: a manager agent coordinates other agents
- How to compare execution patterns
- When to use each process type

By the end, you will have built both types and observed their differences.

## 1. Install & Import

In [ ]:
!pip install -q crewai crewai-tools langchain-openai python-dotenv

## 2. Imports & Setup

In [ ]:
from crewai import Agent, Task, Crew, Processfrom langchain_openai import ChatOpenAIimport osfrom dotenv import load_dotenvload_dotenv()llm = ChatOpenAI(model='gpt-4o-mini', temperature=0.7, api_key=os.getenv('OPENAI_API_KEY'))print('Setup complete')

## 3. Theory: Crews and Processes

### What is a Crew?
A **Crew** is the orchestrator that brings agents and tasks together. It manages:
- Which agents participate
- Which tasks need to be done
- How tasks are executed (the process)
- Memory and context sharing

### Process Types

| Process | Description | Best For |
|---------|-------------|----------|
| **Sequential** | Tasks execute in order, one after another | Linear workflows with clear dependencies |
| **Hierarchical** | A manager agent assigns and reviews tasks | Complex workflows needing oversight |

### Sequential Process
```
Task 1 -> Task 2 -> Task 3 -> Task 4
  Agent A   Agent B   Agent C   Agent D
```

### Hierarchical Process
```
      Manager Agent
           |
    ┌──────┼──────┐
    |      |      |
 Agent A Agent B Agent C
    |      |      |
 Task 1 Task 2 Task 3
    |      |      |
    └──────┼──────┘
           |
      Review & Finalize
```

## 4. Create Agents for Both Crews

In [ ]:
# Create agentsresearcher = Agent(    role='Research Specialist',    goal='Gather accurate and comprehensive information',    backstory='An expert researcher with deep domain knowledge.',    llm=llm,    verbose=True,    allow_delegation=False)writer = Agent(    role='Content Writer',    goal='Create engaging and well-structured content',    backstory='A professional writer who crafts compelling narratives.',    llm=llm,    verbose=True,    allow_delegation=False)editor = Agent(    role='Content Editor',    goal='Polish and improve content quality',    backstory='A meticulous editor who ensures clarity and correctness.',    llm=llm,    verbose=True,    allow_delegation=False)print('Agents created for both crew types')

## 5. Build a Sequential Crew

In a sequential crew, tasks execute in the order they are defined. Each task's output feeds into the next.

In [ ]:
# Define tasks for sequential crewresearch_task = Task(    description='Research the top 5 benefits of remote work for companies in 2025. Be specific.',    expected_output='A list of 5 specific benefits with supporting details.',    agent=researcher)write_task = Task(    description='Write a blog post about remote work benefits using the research provided.',    expected_output='A well-structured blog post of at least 300 words.',    agent=writer,    context=[research_task])edit_task = Task(    description='Edit the blog post for clarity, grammar, and engagement. Improve where needed.',    expected_output='A polished, publication-ready blog post.',    agent=editor,    context=[write_task])# Sequential crewsequential_crew = Crew(    agents=[researcher, writer, editor],    tasks=[research_task, write_task, edit_task],    process=Process.sequential,    verbose=True,    memory=False)print('Sequential crew built: Research -> Write -> Edit')

## 6. Execute the Sequential Crew

In [ ]:
print('\n' + '=' * 60)print('EXECUTING SEQUENTIAL CREW')print('=' * 60)seq_result = sequential_crew.kickoff()print('\n' + '=' * 60)print('SEQUENTIAL CREW OUTPUT')print('=' * 60)print(seq_result)

## 7. Build a Hierarchical Crew

In a hierarchical crew, a manager agent oversees the workflow. Agents can delegate tasks to each other.

In [ ]:
# For hierarchical, agents need allow_delegation=Trueresearcher_h = Agent(    role='Research Specialist',    goal='Gather accurate and comprehensive information',    backstory='An expert researcher.',    llm=llm,    verbose=True,    allow_delegation=True)writer_h = Agent(    role='Content Writer',    goal='Create engaging content',    backstory='A professional writer.',    llm=llm,    verbose=True,    allow_delegation=True)editor_h = Agent(    role='Content Editor',    goal='Polish content quality',    backstory='A meticulous editor.',    llm=llm,    verbose=True,    allow_delegation=True)# Manager agent for hierarchical processmanager = Agent(    role='Project Manager',    goal='Coordinate the team to produce high-quality content efficiently',    backstory=(        'You are an experienced project manager who excels at delegating tasks, '        'reviewing work, and ensuring deliverables meet quality standards. You provide '        'clear instructions and constructive feedback.'    ),    llm=llm,    verbose=True,    allow_delegation=True)# Tasks for hierarchical crew (no explicit context needed - manager handles it)research_task_h = Task(    description='Research the top 5 benefits of remote work for companies in 2025.',    expected_output='A list of 5 specific benefits with supporting details.',    agent=researcher_h)write_task_h = Task(    description='Write a blog post about remote work benefits.',    expected_output='A well-structured blog post of at least 300 words.',    agent=writer_h)edit_task_h = Task(    description='Edit the blog post for clarity and engagement.',    expected_output='A polished, publication-ready blog post.',    agent=editor_h)# Hierarchical crewhierarchical_crew = Crew(    agents=[researcher_h, writer_h, editor_h],    tasks=[research_task_h, write_task_h, edit_task_h],    process=Process.hierarchical,    manager_agent=manager,    verbose=True,    memory=False)print('Hierarchical crew built with Project Manager overseeing workflow')

## 8. Execute the Hierarchical Crew

In [ ]:
print('\n' + '=' * 60)print('EXECUTING HIERARCHICAL CREW')print('=' * 60)hier_result = hierarchical_crew.kickoff()print('\n' + '=' * 60)print('HIERARCHICAL CREW OUTPUT')print('=' * 60)print(hier_result)

## 9. Compare Execution Patterns

| Aspect | Sequential | Hierarchical |
|--------|-----------|--------------|
| Control | Fixed order | Manager decides |
| Flexibility | Low | High |
| Overhead | Low | Higher (manager reviews) |
| Best for | Clear linear workflows | Complex, ambiguous tasks |
| Cost | Lower (fewer LLM calls) | Higher (manager overhead) |

### When to Use Which?
- **Sequential**: Content pipeline, data processing, ETL workflows
- **Hierarchical**: Complex projects, quality-critical tasks, exploratory work

## 10. Crew Configuration Options

```python
Crew(
    agents=[...],           # List of agents
    tasks=[...],            # List of tasks
    process=Process.sequential,  # or Process.hierarchical
    manager_agent=manager,  # Required for hierarchical
    memory=True,            # Enable crew memory
    verbose=True,           # Show execution logs
    max_rpm=10,            # Rate limit API calls
    full_output=False,      # Return full task outputs
    function_calling_llm=llm  # LLM for tool calling
)
```

---

## ✅ Summary

You have successfully:
1. ✅ Built a sequential crew with Research -> Write -> Edit pipeline
2. ✅ Built a hierarchical crew with a manager agent
3. ✅ Executed both and compared their outputs
4. ✅ Understood when to use each process type
5. ✅ Learned crew configuration options

In the next notebook, we will explore **Tools in CrewAI**.